# 🛡️ CyberRott v1.1 — Arquitetura de Treinamento e Auto-Evolução (STaR RL)
### Modelo Especialista em Cibersegurança & Engenharia Reversa com Raciocínio Profundo (`<think>`)
**Plataforma Alvo:** Google Colab Pro / Pro+ com **NVIDIA L4 (24GB VRAM)** ou **A100 (40GB/80GB)**  
**Modelo Base:** `microsoft/Phi-4-mini-instruct` (3.8B, 128k contexto, arquitetura moderna)

---

## 📐 Visão Geral da Arquitetura CyberRott v1.1
O **CyberRott v1.1** implementa uma arquitetura em 3 Fases fundamentadas no pipeline de auto-evolução cognitiva e zero-custo de APIs:

```
+---------------------------------------------------------------------------------------------------+
| FASE 1: Multi-Dataset Anchor Mixing (~10.5k Amostras CoT Open-Source)                            |
| - GooseReason-0.7M (7.5k) + CodeFeedback (1k) + Open-R1 (800) + CyberNative DPO (500) + Sec-v1 (400) |
| - 20x Golden Anchors Ponderadas + Scrubber de Recusas & Padronização CoT 7-Estágios (<think>)     |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 2: Treinamento SFT Base LoRA (Phi-4-mini 3.8B em BF16 / TF32 / SDPA)                        |
| - LoRA r=32, alpha=64, all-linear, Batch Efetivo=16 (batch=2, grad_accum=8, seq_len=2048)        |
| - Gradient Checkpointing ativo (Consumo VRAM: ~16GB na L4 de 24GB sem risco de OOM)               |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 3: Ciclo STaR RL (Self-Taught Reasoner via Rejection Sampling & Verificador Determinístico)  |
| 1. Rollout de Inferência: Geração de N=4 candidatos por problema (tau=0.7)                        |
| 2. Prevenção de OOM na L4: Descarregamento da VRAM (generator.unload() + torch.cuda.empty_cache())|
| 3. STaRVerifier de 3 Pilares: (1) Formato <think>, (2) Ground-Truth / CWE, (3) Sandbox de Código   |
| 4. Rejection Sampling & Ranking: Filtragem e curadoria de trajetórias aprovadas                   |
| 5. Atualização LoRA Incremental: Acúmulo contínuo de aprendizado sobre o checkpoint prévio        |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 4: Auditoria de Vulnerabilidades & Exportação Final (Tripartite Output)                      |
| - Raciocínio CoT (<think>) + Diagnóstico Técnico (FATO) + Correção Defensiva em Sandbox           |
+---------------------------------------------------------------------------------------------------+
```


In [ ]:
# 1. Instalação das dependências modernas (compatibilidade estrita com PyTorch 2.x e CUDA 12)
!pip install -q "transformers>=4.46.0,<5.0.0" "peft>=0.13.0,<0.16.0" "trl>=0.12.0,<0.17.0" "accelerate>=1.0.0" "datasets>=3.0.0" "bitsandbytes>=0.44.0" "pyyaml"

# 2. Diagnóstico de Aceleração de Hardware GPU (L4 / A100)
import os
import torch

print("=" * 70)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    major, minor = torch.cuda.get_device_capability(0)
    bf16_supported = torch.cuda.is_bf16_supported()

    # Ativação de TF32 para arquiteturas modernas (Ampere, Ada Lovelace, Hopper)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print(f"Dispositivo GPU: {device_name}")
    print(f"VRAM Total:      {total_vram_gb:.1f} GB")
    print(f"Arquitetura:     Compute Capability {major}.{minor} (Ampere/Ada: {major >= 8})")
    print(f"Suporte BF16:    {bf16_supported}")
    print(f"TF32 TensorCore: {'Ativado com sucesso' if major >= 8 else 'Não disponível'}")
else:
    print("GPU não detectada! Ative GPU em: Runtime -> Change runtime type -> Hardware accelerator.")
print("=" * 70)
!nvidia-smi


In [ ]:
import os
import sys

# 1. Montagem Opcional do Google Drive (Persistência Segura de Checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DRIVE_DIR = "/content/drive/MyDrive/CyberRott_Checkpoints"
    os.makedirs(CHECKPOINT_DRIVE_DIR, exist_ok=True)
    print(f"[✓] Google Drive montado com sucesso em: {CHECKPOINT_DRIVE_DIR}")
except Exception:
    print("[*] Google Drive não montado (ambiente local ou execução independente).")

# 2. Localização e Sincronização do Repositório CyberRott1.0
REPO_DIR = "/content/CyberRott1.0"
if os.path.exists("/content"):
    if not os.path.exists(REPO_DIR):
        print("[*] Clonando repositório CyberRott1.0 no ambiente Colab...")
        !git clone https://github.com/johnknevescarrijo/CyberRott1.0.git {REPO_DIR}
    else:
        print("[*] Repositório CyberRott1.0 já existente. Atualizando via git pull...")
        !git -C {REPO_DIR} pull origin main
    os.chdir(REPO_DIR)
elif os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
    os.chdir(REPO_DIR)

print(f"[✓] Diretório de Trabalho: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

# 3. Configuração do HF_TOKEN (para datasets públicos do Hugging Face Hub)
os.environ["HF_TOKEN"] = "hf_" + "qxdgNJSKXSlJTnvDtwozTpfQoWbhoeliWP"
print("[✓] HF_TOKEN configurado com sucesso.")


---
## 📦 FASE 1: Data Prep Multi-Dataset v1.1 (Zero-API Cost Data Expansion)

Nesta fase, executamos o script unificado `scripts/data_prep.py` com o preset `--preset v1.1`.
Ele extrai e funde em streaming 5 corpora abertos de cibersegurança e raciocínio formal (~10.2k amostras) + 20x upsampling das Amostras Âncora de Ouro complexas:
- **GooseReason-0.7M** (7.5k amostras de dedução lógica e algoritmos)
- **CodeFeedback-Filtered** (1.0k amostras de auditoria de código e memória)
- **Open-R1 / Bespoke-Stratos** (800 trajetórias long-CoT)
- **CyberNative Vulnerability DPO** (500 pares de CWEs e patches seguros)
- **AlicanKiraz0 Cybersecurity v1** (400 amostras Red/Blue Team e MITRE ATT&CK)

Aplica o **Scrubber Programático** de recusas alinhadas ("As an AI...", "I cannot assist...") e padroniza o raciocínio CoT de 7 estágios em tags nativas `<think>...</think>`.


In [ ]:
import os
import json

DATASET_FILE = "data/cyberrott_final_train.jsonl"

def count_samples(filepath):
    if not os.path.exists(filepath):
        return 0
    with open(filepath, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

# Checagem: se o dataset v1.1 completo já existir e tiver >= 4500 amostras, pula o download/prep
existing_samples = count_samples(DATASET_FILE)
if existing_samples >= 4500:
    print("[✓] Dataset v1.1 completo (4.680 amostras / 40 MB) já presente localmente via Git! Pulando download...")
else:
    print(f"Dataset local ausente ou incompleto ({existing_samples} amostras). Executando pipeline data_prep.py...")
    !python scripts/data_prep.py \
        --preset v1.1 \
        --anchor-file data/dataset_cyber_ready.jsonl \
        --output-file data/cyberrott_final_train.jsonl \
        --seed 42

# Validação e Estatísticas do Corpus Gerado
if os.path.exists(DATASET_FILE):
    size_mb = os.path.getsize(DATASET_FILE) / (1024 * 1024)
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        total_samples = sum(1 for line in f if line.strip())
    print(f"\n[✓] Corpus final gerado com sucesso: {total_samples:,} amostras ({size_mb:.2f} MB)")

    # Visualização de amostra aleatória
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        sample = json.loads(f.readline())
    print("\n" + "=" * 60)
    print("EXEMPLO DE AMOSTRA CURADA (ChatML):")
    print("=" * 60)
    for msg in sample.get("messages", []):
        snippet = msg['content'][:250] + ("..." if len(msg['content']) > 250 else "")
        print(f"[{msg['role'].upper()}]:\n{snippet}\n")
else:
    print(f"[Erro] Arquivo {DATASET_FILE} não foi gerado.")


---
## 🚀 FASE 2: Treinamento SFT Base LoRA v1.1

Fine-tuning supervisionado do modelo **Microsoft Phi-4-mini-instruct (3.8B)** utilizando TRL `SFTTrainer` e PEFT `LoraConfig`:
- **Precisão:** `bfloat16` nativo com aceleração TF32 em Tensor Cores
- **Atenção:** `sdpa` (PyTorch Scaled Dot-Product Attention) ou `flash_attention_2`
- **Configuração LoRA:** `r=32`, `alpha=64`, dropout `0.05`, módulos `all-linear` (projeta todas as matrizes QKV, saída e MLP)
- **Hiperparâmetros de Memória para L4 (24GB):**
  - `per_device_train_batch_size = 2`
  - `gradient_accumulation_steps = 8` (Batch size efetivo = 16)
  - `gradient_checkpointing = True`
  - Consumo máximo de VRAM: ~16GB a 18GB, garantindo estabilidade total contra OOM.


In [ ]:
# Executa o Treinamento SFT LoRA da Fase 2
!python scripts/train_colab.py \
    --data_path data/cyberrott_final_train.jsonl \
    --output_dir ./cyberrott_v1_1_sft_lora \
    --epochs 3 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_seq_length 2048 \
    --lora_r 32 \
    --lora_alpha 64 \
    --torch_dtype bfloat16 \
    --attn_implementation sdpa \
    --learning_rate 2e-4


---
## 🧠 FASE 3: Ciclo STaR RL (Self-Taught Reasoner via Rejection Sampling)

O algoritmo **STaR** implementa auto-evolução cognitiva em ciclo fechado:
1. **Rollout Exploratório:** O checkpoint SFT da Fase 2 gera $N=4$ trajetórias analíticas por problema com temperatura $\tau=0.7$.
2. **Prevenção de OOM na L4:** O método `generator.unload()` descarrega imediatamente os tensores da VRAM e chama `gc.collect()` + `torch.cuda.empty_cache()` antes de invocar o `SFTTrainer`.
3. **STaRVerifier Determinístico de 3 Pilares:**
   - **Pilar 1 (Formato & Refusal):** Integridade de `<think>...</think>`, sem recusas conversacionais e formato de saída tripartido.
   - **Pilar 2 (Ground-Truth & CWE):** Correspondência estrita de gabarito e identificação precisa de CWE.
   - **Pilar 3 (Code Sandbox):** Validação de sintaxe via AST Python, analisador de código C e sintaxe Bash.
4. **Rejection & Ranking:** Candidatos reprovados são descartados sumariamente; os aprovados são pontuados e fundidos com as Âncoras de Ouro.
5. **Atualização LoRA Incremental (`run_sft_update`):** O modelo acumula aprendizado contínuo sobre o checkpoint anterior sem falhas no carregamento de adaptadores.


In [ ]:
# Executa o Ciclo de Auto-Evolução STaR RL
!python scripts/train_star.py \
    --model_path ./cyberrott_v1_1_sft_lora \
    --problem_file data/dataset_cyber_ready.jsonl \
    --anchor_file data/dataset_cyber_ready.jsonl \
    --output_dir ./cyberrott_v1_1_star_rl \
    --iterations 2 \
    --num_candidates 4 \
    --backend hf \
    --max_seq_length 2048 \
    --lora_r 32 \
    --lora_alpha 64 \
    --torch_dtype bfloat16 \
    --attn_implementation sdpa


---
## 🎯 FASE 4: Inferência, Auditoria e Backup do Modelo Final

Nesta fase final:
1. Carregamos o modelo auto-evoluído da Fase 3 (`checkpoint-iter-2` ou fallback).
2. Testamos um desafio complexo de vulnerabilidade em código C/C++ verificando a saída tripartite:
   - `<think>...</think>` (Cadeia analítica de 7 estágios: superfície, fluxo, sink, CWE, impacto)
   - `FATO: ...` (Diagnóstico conclusivo sem recusa conversacional)
   - `SUGESTÃO DE CORREÇÃO:` (Patch defensivo funcional com bounds checking)
3. Sincronizamos os checkpoints gerados com o Google Drive para persistência de longo prazo.


In [ ]:
import os
import shutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Identificar o Checkpoint Final
FINAL_CHECKPOINT = "./cyberrott_v1_1_star_rl/checkpoint-iter-2"
if not os.path.exists(FINAL_CHECKPOINT):
    for candidate in ["./cyberrott_v1_1_star_rl/checkpoint-iter-1", "./cyberrott_v1_1_sft_lora"]:
        if os.path.exists(candidate):
            FINAL_CHECKPOINT = candidate
            break

print(f"[*] Checkpoint Selecionado: {FINAL_CHECKPOINT}")

BASE_MODEL = "microsoft/Phi-4-mini-instruct"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

# Carrega Tokenizer e Modelo Base em bfloat16
tokenizer = AutoTokenizer.from_pretrained(
    FINAL_CHECKPOINT if os.path.exists(os.path.join(FINAL_CHECKPOINT, "tokenizer_config.json")) else BASE_MODEL,
    trust_remote_code=False
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    trust_remote_code=False,
)

# Mescla pesos LoRA com o modelo base para inferência direta
print(f"[*] Mesclando pesos adaptadores de {FINAL_CHECKPOINT}...")
model = PeftModel.from_pretrained(base_model, FINAL_CHECKPOINT).merge_and_unload()
model.eval()
print("[✓] Modelo CyberRott pronto para auditoria técnica!")

# 2. Desafio de Auditoria em C (Buffer Overflow / Taint Analysis)
audit_prompt = (
    "Realize a auditoria de segurança rigorosa da seguinte função C em espaço de kernel/usuário:\n\n"
    "```c\n"
    "void parse_network_packet(const char *incoming_payload, int packet_len) {\n"
    "    char local_buffer[64];\n"
    "    if (packet_len > 0) {\n"
    "        strcpy(local_buffer, incoming_payload);\n"
    "    }\n"
    "    process_packet_header(local_buffer);\n"
    "}\n"
    "```\n\n"
    "Instruções:\n"
    "1. Desenvolva seu raciocínio analítico dentro da tag <think>...</think>.\n"
    "2. Declare a vulnerabilidade identificada, o CWE correspondente e o impacto.\n"
    "3. Forneça o patch corretivo defensivo com verificação rígida de limites."
)

messages = [
    {
        "role": "system",
        "content": "Você é o CyberRott, uma IA especialista em segurança ofensiva, defensiva e análise de vulnerabilidades com raciocínio analítico profundo."
    },
    {"role": "user", "content": audit_prompt}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

print("\n" + "=" * 70)
print("🛡️ RESPOSTA DO CYBERROTT (RACIOCÍNIO TRIPARTITE):")
print("=" * 70 + "\n")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

input_len = inputs["input_ids"].shape[1]
output_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(output_text)

# 3. Backup para o Google Drive
DRIVE_BACKUP = "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_1_Final"
if os.path.exists("/content/drive/MyDrive"):
    print(f"\n[*] Sincronizando checkpoint final para o Google Drive: {DRIVE_BACKUP}...")
    shutil.copytree(FINAL_CHECKPOINT, DRIVE_BACKUP, dirs_exist_ok=True)
    print("[✓] Backup no Google Drive concluído com sucesso!")
